# 06 — Recoverability Under Partial Observation

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** corrupt, hide, or mix prime data with noise, then test which constraint structures remain recoverable.

Notebook 05 showed that random controls can match count without recovering structure.  
Notebook 06 tests whether damaged prime observations still contain recoverable constraint signal.

Core claim:

> Partial observation weakens count recovery but does not erase constraint signal.

Clean phrase:

> Constraint structure remains detectable after corruption.

## 0. Setup

Artifact structure:

```text
06_recoverability_under_partial_observation/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
06_recoverability_under_partial_observation_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "06_recoverability_under_partial_observation"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Recoverability Under Partial Observation"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Start with the true prime set:

\[
P_N = \{p : p \le N\}
\]

Then create corrupted observations:

1. **Missing-prime observations**  
   Keep only a fraction of primes.

2. **Noisy observations**  
   Add random composite / non-prime points.

3. **Mixed corruption**  
   Remove primes and add noise.

Then measure whether structure remains detectable.

## 2. Metrics

### Recovery

\[
CGCS_{\mathrm{recovery}}(S)=
\frac{|S \cap P_N|}{|P_N|}
\]

### Precision

\[
precision(S)=
\frac{|S \cap P_N|}{|S|}
\]

### Mod 6 constraint score

For \(S_{>3}\):

\[
CGCS_{\mathrm{mod6}}(S)=
\frac{|\{n\in S:n\equiv 1,5\pmod6\}|}{|S_{>3}|}
\]

### Density drift

\[
drift_{\mathrm{density}}(S)=
\operatorname{mean}_x
\frac{||S\cap[2,x]|-\pi(x)|}{\pi(x)}
\]

### Sieve-consistency score

\[
CGCS_{\mathrm{sieve}}(S)=
\frac{|\{n\in S:n\text{ passes primality test}\}|}{|S|}
\]

This measures how prime-like the observed set is.

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

KEEP_FRACTIONS = [1.00, 0.90, 0.75, 0.50, 0.25]
NOISE_FRACTIONS = [0.10, 0.25, 0.50, 1.00]
MIXED_CONFIGS = [
    {"keep_fraction": 0.75, "noise_fraction": 0.25},
    {"keep_fraction": 0.50, "noise_fraction": 0.50},
    {"keep_fraction": 0.25, "noise_fraction": 1.00},
]

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "KEEP_FRACTIONS": KEEP_FRACTIONS,
    "NOISE_FRACTIONS": NOISE_FRACTIONS,
    "MIXED_CONFIGS": MIXED_CONFIGS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Generate reference primes

Use a standard sieve for reference structure.

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
prime_set = set(reference_primes.tolist())

universe = np.arange(2, N_MAX + 1)
composites = np.array([n for n in universe if n not in prime_set], dtype=int)

summary = {
    "n_max": int(N_MAX),
    "universe_count": int(len(universe)),
    "prime_count": int(len(reference_primes)),
    "composite_count": int(len(composites)),
    "first_primes": reference_primes[:10].tolist(),
    "last_primes": reference_primes[-10:].tolist(),
}

summary

## 4. Create corrupted observations

Generate:

- missing-prime observations
- noisy observations
- mixed observations

In [ ]:
observations = []

# Baseline true observation.
observations.append({
    "name": "true_primes",
    "scenario": "baseline",
    "keep_fraction": 1.0,
    "noise_fraction": 0.0,
    "values": reference_primes.copy(),
})

# Missing-prime observations.
for keep_fraction in KEEP_FRACTIONS:
    keep_count = int(round(keep_fraction * len(reference_primes)))
    values = np.sort(rng.choice(reference_primes, size=keep_count, replace=False))
    observations.append({
        "name": f"missing_keep_{int(keep_fraction*100)}",
        "scenario": "missing",
        "keep_fraction": keep_fraction,
        "noise_fraction": 0.0,
        "values": values,
    })

# Noisy observations: true primes plus random composites.
for noise_fraction in NOISE_FRACTIONS:
    noise_count = int(round(noise_fraction * len(reference_primes)))
    noise_values = rng.choice(composites, size=noise_count, replace=False)
    values = np.sort(np.unique(np.concatenate([reference_primes, noise_values])))
    observations.append({
        "name": f"noise_{int(noise_fraction*100)}",
        "scenario": "noise",
        "keep_fraction": 1.0,
        "noise_fraction": noise_fraction,
        "values": values,
    })

# Mixed observations.
for cfg in MIXED_CONFIGS:
    keep_fraction = cfg["keep_fraction"]
    noise_fraction = cfg["noise_fraction"]
    keep_count = int(round(keep_fraction * len(reference_primes)))
    noise_count = int(round(noise_fraction * len(reference_primes)))

    kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
    noise_values = rng.choice(composites, size=noise_count, replace=False)

    values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))
    observations.append({
        "name": f"mixed_keep_{int(keep_fraction*100)}_noise_{int(noise_fraction*100)}",
        "scenario": "mixed",
        "keep_fraction": keep_fraction,
        "noise_fraction": noise_fraction,
        "values": values,
    })

[(obs["name"], obs["scenario"], len(obs["values"])) for obs in observations]

## 5. Metric functions

Measure recovery, precision, mod 6 score, density drift, and sieve-consistency.

In [ ]:
def is_prime_array(values: np.ndarray, prime_set: set) -> np.ndarray:
    return np.array([int(v) in prime_set for v in values], dtype=bool)

def mod6_score(values: np.ndarray) -> float:
    gt3 = values[values > 3]
    if len(gt3) == 0:
        return float("nan")
    return float(np.mean(np.isin(gt3 % 6, [1, 5])))

def density_drift(values: np.ndarray, scales: np.ndarray) -> tuple[float, pd.DataFrame]:
    values = np.sort(values)
    rows = []
    drifts = []
    for x in scales:
        obs_count = int(np.searchsorted(values, x, side="right"))
        prime_count = int(np.searchsorted(reference_primes, x, side="right"))
        drift = abs(obs_count - prime_count) / prime_count if prime_count else 0.0
        drifts.append(drift)
        rows.append({
            "x": int(x),
            "observed_count": obs_count,
            "prime_count": prime_count,
            "density_drift": float(drift),
        })
    return float(np.mean(drifts)), pd.DataFrame(rows)

def metrics_for_observation(obs: dict, scales: np.ndarray) -> tuple[dict, pd.DataFrame, dict]:
    values = np.sort(obs["values"])
    value_set = set(values.tolist())
    is_prime_mask = is_prime_array(values, prime_set)

    true_overlap = int(is_prime_mask.sum())
    false_positive_count = int((~is_prime_mask).sum())
    false_negative_count = int(len(prime_set - value_set))

    recovery = true_overlap / len(reference_primes)
    precision = true_overlap / len(values) if len(values) else float("nan")
    recovery_drift = 1.0 - recovery
    mod6 = mod6_score(values)
    sieve_consistency = precision

    mean_density_drift, density_curve = density_drift(values, scales)
    density_curve["name"] = obs["name"]
    density_curve["scenario"] = obs["scenario"]

    # Reconstruction: filter observed values through primality / sieve-consistency.
    reconstructed = values[is_prime_mask]
    reconstructed_set = set(reconstructed.tolist())
    reconstruction_recovery = len(reconstructed_set & prime_set) / len(reference_primes)
    reconstruction_precision = 1.0 if len(reconstructed) else float("nan")

    metric = {
        "name": obs["name"],
        "scenario": obs["scenario"],
        "keep_fraction": float(obs["keep_fraction"]),
        "noise_fraction": float(obs["noise_fraction"]),
        "observed_count": int(len(values)),
        "true_prime_overlap": true_overlap,
        "false_positive_count": false_positive_count,
        "false_negative_count": false_negative_count,
        "cgcs_recovery": float(recovery),
        "precision": float(precision),
        "recovery_drift": float(recovery_drift),
        "cgcs_mod6": float(mod6),
        "mean_density_drift": float(mean_density_drift),
        "cgcs_sieve_consistency": float(sieve_consistency),
        "reconstruction_recovery": float(reconstruction_recovery),
        "reconstruction_precision": float(reconstruction_precision),
    }

    reconstruction = {
        "name": obs["name"],
        "scenario": obs["scenario"],
        "observed_count": int(len(values)),
        "reconstructed_count": int(len(reconstructed)),
        "reconstruction_recovery": float(reconstruction_recovery),
        "reconstruction_precision": float(reconstruction_precision),
    }

    return metric, density_curve, reconstruction

scales = np.unique(np.logspace(2, np.log10(N_MAX), 70).astype(int))

metric_rows = []
density_frames = []
reconstruction_rows = []

for obs in observations:
    metric, density_curve, reconstruction = metrics_for_observation(obs, scales)
    metric_rows.append(metric)
    density_frames.append(density_curve)
    reconstruction_rows.append(reconstruction)

observation_metrics_df = pd.DataFrame(metric_rows)
density_drift_df = pd.concat(density_frames, ignore_index=True)
reconstruction_metrics_df = pd.DataFrame(reconstruction_rows)

observation_metrics_df.head(), reconstruction_metrics_df.head()

## 6. Summary scores

Main expectation:

- missing primes reduce recovery but preserve precision and mod 6 structure
- added noise reduces precision and sieve-consistency
- mixed corruption weakens both recovery and precision

In [ ]:
baseline = observation_metrics_df[observation_metrics_df["name"] == "true_primes"].iloc[0]

best_noisy = observation_metrics_df[observation_metrics_df["scenario"] == "noise"].sort_values("precision", ascending=False).iloc[0]
worst_mixed = observation_metrics_df[observation_metrics_df["scenario"] == "mixed"].sort_values("cgcs_recovery").iloc[0]

measurement = {
    "baseline_recovery": float(baseline["cgcs_recovery"]),
    "baseline_precision": float(baseline["precision"]),
    "baseline_mod6": float(baseline["cgcs_mod6"]),
    "best_noisy_precision": float(best_noisy["precision"]),
    "worst_mixed_recovery": float(worst_mixed["cgcs_recovery"]),
    "worst_mixed_precision": float(worst_mixed["precision"]),
    "mean_recovery_all_observations": float(observation_metrics_df["cgcs_recovery"].mean()),
    "mean_precision_all_observations": float(observation_metrics_df["precision"].mean()),
    "mean_mod6_all_observations": float(observation_metrics_df["cgcs_mod6"].mean()),
    "mean_sieve_consistency_all_observations": float(observation_metrics_df["cgcs_sieve_consistency"].mean()),
}

cgcs = {
    "score": float(baseline["cgcs_recovery"]),
    "definition": "CGCS_recovery(S)=|S∩P_N|/|P_N|",
    "interpretation": "Recovery decreases with missing observations; precision and sieve-consistency detect noise.",
}

measurement

## 7. Figure 1 — recovery vs keep fraction

Missing-prime observations reduce recovery as keep fraction decreases.

In [ ]:
missing_df = observation_metrics_df[observation_metrics_df["scenario"].isin(["baseline", "missing"])].copy()
missing_df = missing_df.sort_values("keep_fraction")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(missing_df["keep_fraction"], missing_df["cgcs_recovery"], marker="o")
ax.set_title("Recovery vs keep fraction")
ax.set_xlabel("keep fraction")
ax.set_ylabel("CGCS recovery")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_recovery_vs_keep_fraction.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 8. Figure 2 — precision vs noise level

Added composite noise reduces precision and sieve-consistency.

In [ ]:
noise_df = observation_metrics_df[observation_metrics_df["scenario"].isin(["baseline", "noise"])].copy()
noise_df = noise_df.sort_values("noise_fraction")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(noise_df["noise_fraction"], noise_df["precision"], marker="o", label="precision")
ax.plot(noise_df["noise_fraction"], noise_df["cgcs_sieve_consistency"], marker="o", label="sieve-consistency")
ax.set_title("Precision vs noise level")
ax.set_xlabel("noise fraction")
ax.set_ylabel("score")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_precision_vs_noise.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 9. Figure 3 — mod 6 score by scenario

Residue constraint remains detectable when primes are missing.  
Noise weakens the score.

In [ ]:
scenario_mod6 = (
    observation_metrics_df
    .groupby("scenario", observed=True)
    .agg(mean_mod6=("cgcs_mod6", "mean"), min_mod6=("cgcs_mod6", "min"), max_mod6=("cgcs_mod6", "max"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(scenario_mod6["scenario"], scenario_mod6["mean_mod6"])
ax.set_ylim(0, 1.05)
ax.set_title("Mean mod 6 score by scenario")
ax.set_xlabel("scenario")
ax.set_ylabel("CGCS mod 6")
ax.grid(True, axis="y", alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_mod6_score_by_scenario.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

scenario_mod6, fig3_path

## 10. Figure 4 — density drift by observation

Density drift records count-scale damage.

In [ ]:
plot_df = observation_metrics_df.copy()
plot_df["label"] = plot_df["name"].str.replace("_", "\n")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df["label"], plot_df["mean_density_drift"])
ax.set_title("Density drift by observation")
ax.set_xlabel("observation")
ax.set_ylabel("mean density drift")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_density_drift_by_scenario.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

## 11. Figure 5 — sieve-consistency score

Sieve-consistency detects noise directly.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df["label"], plot_df["cgcs_sieve_consistency"])
ax.set_ylim(0, 1.05)
ax.set_title("Sieve-consistency score by observation")
ax.set_xlabel("observation")
ax.set_ylabel("CGCS sieve-consistency")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_sieve_consistency_scores.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

## 12. Figure 6 — reconstruction scores

Filter noisy observations through sieve-consistency, then measure recovered precision and recovery.

In [ ]:
recon_plot = reconstruction_metrics_df.copy()
recon_plot["label"] = recon_plot["name"].str.replace("_", "\n")

x = np.arange(len(recon_plot))
width = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, recon_plot["reconstruction_recovery"], width, label="reconstruction recovery")
ax.bar(x + width/2, recon_plot["reconstruction_precision"], width, label="reconstruction precision")
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(recon_plot["label"], rotation=45, ha="right")
ax.set_title("Reconstruction scores after sieve-consistency filtering")
ax.set_xlabel("observation")
ax.set_ylabel("score")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_reconstruction_scores.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

## 13. Interpretation

1. **Missing data:** recovery decreases, but precision and mod 6 structure remain strong.

2. **Noisy data:** recovery can remain high, but precision and sieve-consistency decrease.

3. **Mixed corruption:** both recovery and precision weaken.

4. **Reconstruction:** sieve-consistency filtering removes noise and recovers precision, but cannot restore missing primes.

Core statement:

> Constraint structure remains detectable after corruption.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook corrupted the prime set through missing observations, added noise, and mixed corruption.",
    "",
    "The goal was to test which constraint signals remain recoverable under partial observation.",
    "",
    "## Main findings",
    "",
    "Partial observation weakens count recovery but does not erase constraint signal.",
    "",
    "Missing-prime observations preserve precision and mod 6 structure while reducing recovery.",
    "",
    "Noisy observations preserve recovery when all primes remain present, but precision and sieve-consistency decline.",
    "",
    "Mixed corruption weakens both recovery and precision.",
    "",
    "## Recovery and precision",
    "",
    f"- baseline recovery = {measurement['baseline_recovery']:.6f}",
    f"- baseline precision = {measurement['baseline_precision']:.6f}",
    f"- worst mixed recovery = {measurement['worst_mixed_recovery']:.6f}",
    f"- worst mixed precision = {measurement['worst_mixed_precision']:.6f}",
    "",
    "## Mod 6 and sieve-consistency",
    "",
    "The mod 6 score remains high for missing-prime observations because removing primes does not add invalid residues.",
    "",
    "Added composite noise lowers precision and sieve-consistency.",
    "",
    "## Reconstruction",
    "",
    "Sieve-consistency filtering removes composite noise and restores precision among retained observations.",
    "",
    "It cannot restore missing primes, so reconstruction recovery remains limited by the keep fraction.",
    "",
    "## Core result",
    "",
    "Constraint structure remains detectable after corruption.",
    "",
    "## Caution",
    "",
    "This notebook studies finite recoverability diagnostics. It does not prove a new theorem about primes.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path]
figure_titles = [
    "Recovery vs keep fraction",
    "Precision vs noise level",
    "Mod 6 score by scenario",
    "Density drift by observation",
    "Sieve-consistency scores",
    "Reconstruction scores",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 14. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
observation_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_observation_metrics.csv"
density_drift_path = DATA_DIR / f"{NOTEBOOK_NUM}_density_drift.csv"
reconstruction_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_reconstruction_metrics.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
observation_metrics_df.to_csv(observation_metrics_path, index=False)
density_drift_df.to_csv(density_drift_path, index=False)
reconstruction_metrics_df.to_csv(reconstruction_metrics_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "observation_metrics": str(observation_metrics_path),
        "density_drift": str(density_drift_path),
        "reconstruction_metrics": str(reconstruction_metrics_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 06 follows Notebook 05 by testing whether corrupted prime observations still contain recoverable constraint signal.",
    "",
    "Notebook 05 showed random controls fail to recover structure.",
    "Notebook 06 shows that damaged structure can still retain recoverable signal.",
    "",
    "## Corruption scenarios",
    "",
    "1. missing-prime observations",
    "2. noisy observations",
    "3. mixed corruption",
    "",
    "## Measurements",
    "",
    "1. recovery",
    "2. precision",
    "3. mod 6 constraint score",
    "4. density drift",
    "5. sieve-consistency",
    "6. reconstruction after sieve-consistency filtering",
    "",
    "## Core claim",
    "",
    "Partial observation weakens count recovery but does not erase constraint signal.",
    "",
    "## Figures",
    "",
    "1. recovery vs keep fraction",
    "2. precision vs noise level",
    "3. mod 6 score by scenario",
    "4. density drift by observation",
    "5. sieve-consistency scores",
    "6. reconstruction scores",
    "",
    "## Handoff",
    "",
    "Notebook 07 should reconstruct candidate prime structure from corrupted observations using constraints.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook corrupts prime observations and measures which constraint signals remain recoverable.",
    "",
    r"The recovery score is",
    r"\[",
    r"CGCS_{\mathrm{recovery}}(S)=",
    r"\frac{|S\cap P_N|}{|P_N|}.",
    r"\]",
    "",
    r"The precision score is",
    r"\[",
    r"precision(S)=",
    r"\frac{|S\cap P_N|}{|S|}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item baseline recovery $= {measurement['baseline_recovery']:.6f}$",
    rf"  \item baseline precision $= {measurement['baseline_precision']:.6f}$",
    rf"  \item worst mixed recovery $= {measurement['worst_mixed_recovery']:.6f}$",
    rf"  \item worst mixed precision $= {measurement['worst_mixed_precision']:.6f}$",
    r"\end{itemize}",
    "",
    r"Constraint structure remains detectable after corruption.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Recoverability Under Partial Observation}",
    "",
    r"\subsection*{Prime set}",
    "",
    r"\[",
    r"P_N = \{p : p \le N\}.",
    r"\]",
    "",
    r"\subsection*{Recovery}",
    "",
    r"\[",
    r"CGCS_{\mathrm{recovery}}(S)=",
    r"\frac{|S \cap P_N|}{|P_N|}.",
    r"\]",
    "",
    r"\subsection*{Precision}",
    "",
    r"\[",
    r"precision(S)=",
    r"\frac{|S \cap P_N|}{|S|}.",
    r"\]",
    "",
    r"\subsection*{Mod 6 score}",
    "",
    r"\[",
    r"CGCS_{\mathrm{mod6}}(S)=",
    r"\frac{|\{n\in S:n\equiv 1,5\pmod6\}|}{|S_{>3}|}.",
    r"\]",
    "",
    r"\subsection*{Density drift}",
    "",
    r"\[",
    r"drift_{\mathrm{density}}(S)=",
    r"\operatorname{mean}_x",
    r"\frac{||S\cap[2,x]|-\pi(x)|}{\pi(x)}.",
    r"\]",
    "",
    r"\subsection*{Sieve-consistency}",
    "",
    r"\[",
    r"CGCS_{\mathrm{sieve}}(S)=",
    r"\frac{|\{n\in S:n\text{ passes prime divisibility tests}\}|}{|S|}.",
    r"\]",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, observation_metrics_path, density_drift_path, reconstruction_metrics_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 15. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 16. Next notebook handoff

Next notebook:

```text
07_constraint_reconstruction.ipynb
```

Purpose:

> use only observed corrupted samples to reconstruct candidate prime structure through residue, density, and sieve-consistency constraints.

In [ ]:
next_step = "Notebook 07: constraint reconstruction from corrupted observations."
print(next_step)